# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [39]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
    solve_full_grid_fc_one_pass
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [40]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])


5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [41]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [42]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [43]:
# Attempt (a): solve the puzzle with pl_resolution / tt_entails on the general KB.
#
# The instructions say to uncomment this and hit Kernel > Interrupt after ~30s.
# We run it under a guard instead, so the notebook can be submitted with its
# output visible. Two wrinkles made the guard more than just a timer:
#
#   * pl_resolution builds its entire list of clause pairs before resolving any
#     of them, so on this KB it passes 4 GB of RAM in about ten seconds. It gets
#     a memory budget on top of the time budget. tt_entails does not need one --
#     it fails on time, not space -- and giving it one only hides how it fails.
#   * A Python thread cannot be cancelled from outside, so a thread we stopped
#     waiting on would keep allocating while the rest of the notebook runs. We
#     raise an exception inside it to actually stop it.

import os
import sys
import time
import ctypes
import threading

try:
    import psutil
except ImportError:
    psutil = None

TIME_BUDGET = 20     # seconds, roughly the ~30s the instructions suggest
MEMORY_BUDGET = 800  # extra MB pl_resolution may allocate before we stop it


class GaveUp(BaseException):
    """Raised inside a worker thread we have stopped waiting on."""


# A stopped thread unwinds with GaveUp, which Python otherwise reports as a long
# traceback. Silence just that one exception so the output stays readable.
_thread_excepthook, _unraisable_hook = threading.excepthook, sys.unraisablehook


def _quiet(args, fallback):
    if args.exc_type is not GaveUp:
        fallback(args)


threading.excepthook = lambda a: _quiet(a, _thread_excepthook)
sys.unraisablehook = lambda a: _quiet(a, _unraisable_hook)


def try_inference(label, fn, time_limit=TIME_BUDGET, mem_limit=None):
    """Run fn() on a background thread and report what happened.

    Stops at whichever comes first: a returned value, an exception, the time
    budget, or the memory budget if one was given. Returns a one-line summary.
    """
    out = {}

    def target():
        try:
            out['value'] = fn()
        except Exception as e:      # RecursionError turns up here
            out['error'] = f'{type(e).__name__}: {e}'

    def give_up(reason):
        if thread.is_alive():
            ctypes.pythonapi.PyThreadState_SetAsyncExc(
                ctypes.c_long(thread.ident), ctypes.py_object(GaveUp))
            thread.join(5)
        return f'{label}: gave up, {reason}'

    watch_memory = psutil is not None and mem_limit is not None
    proc = psutil.Process(os.getpid()) if watch_memory else None
    base = proc.memory_info().rss / 1e6 if watch_memory else 0

    thread = threading.Thread(target=target, daemon=True)
    start = time.time()
    thread.start()

    while thread.is_alive():
        thread.join(0.5)
        elapsed = time.time() - start
        if watch_memory:
            used = proc.memory_info().rss / 1e6 - base
            if used > mem_limit:
                return give_up(f'had allocated {used:.0f} MB after {elapsed:.0f}s')
        if elapsed > time_limit:
            return give_up(f'still running after {time_limit}s')

    elapsed = time.time() - start
    if 'error' in out:
        return f'{label}: raised {out["error"]} after {elapsed:.1f}s'
    return f'{label}: returned {out["value"]} in {elapsed:.2f}s'


# How big a problem are we handing these two algorithms?
general_kb = build_general_kb(n, box_h, box_w, givens)
symbols = set()
for clause in general_kb.clauses:
    symbols |= prop_symbols(clause)

num_clauses = len(general_kb.clauses)
num_pairs = num_clauses * (num_clauses - 1) // 2
print(f'general KB: {num_clauses} clauses over {len(symbols)} symbols')
print(f'  tt_entails enumerates 2^{len(symbols)} models')
print(f'  pl_resolution compares {num_pairs:,} clause pairs in round 1 alone')
print()

# Cell (1,1) is empty in this puzzle and its solved value is 1, so Is1_1_1 is
# genuinely entailed -- a correct algorithm has to return True here.
r, c, v = 1, 1, givens.get((1, 1), 1)
query = atom('Is', r, c, v)
print(f'query: {query}   (cell ({r},{c}) is empty; expected answer True)')

if psutil:
    print(try_inference('pl_resolution', lambda: pl_resolution(general_kb, query),
                        mem_limit=MEMORY_BUDGET))
else:
    print('pl_resolution: skipped, needs psutil for the memory guard (pip install psutil)')
print(try_inference('tt_entails', lambda: tt_entails(associate('&', general_kb.clauses), query)))

general KB: 11775 clauses over 729 symbols
  tt_entails enumerates 2^729 models
  pl_resolution compares 69,319,425 clause pairs in round 1 alone

query: Is1_1_1   (cell (1,1) is empty; expected answer True)
pl_resolution: gave up, had allocated 871 MB after 2s
tt_entails: gave up, still running after 20s


In [44]:
# Where does the wall actually show up? Same build_general_kb, same two
# algorithms, just smaller boards. This separates "our encoding is broken" from
# "this does not scale": if both algorithms answer correctly on a small board,
# then the encoding is fine and size is the only thing going wrong.

# 2x2 board, 1x2 boxes. Solution is  1 2 / 2 1, so Is2_2_1 is entailed.
tiny_givens = {(1, 1): 1}

# 4x4 board, 2x2 boxes. Solution is 1234 / 3412 / 2143 / 4321. We blank out the
# four cells on the diagonal, so Is1_1_1 is entailed but has to be inferred.
small_solution = {
    (1, 1): 1, (1, 2): 2, (1, 3): 3, (1, 4): 4,
    (2, 1): 3, (2, 2): 4, (2, 3): 1, (2, 4): 2,
    (3, 1): 2, (3, 2): 1, (3, 3): 4, (3, 4): 3,
    (4, 1): 4, (4, 2): 3, (4, 3): 2, (4, 4): 1,
}
small_givens = {rc: val for rc, val in small_solution.items() if rc[0] != rc[1]}

boards = [
    ('2x2 board (1x2 boxes)', 2, 1, 2, tiny_givens, atom('Is', 2, 2, 1)),
    ('4x4 board (2x2 boxes)', 4, 2, 2, small_givens, atom('Is', 1, 1, 1)),
]

for label, size, bh, bw, board_givens, board_query in boards:
    kb = build_general_kb(size, bh, bw, board_givens)
    syms = set()
    for clause in kb.clauses:
        syms |= prop_symbols(clause)
    print(f'{label}: {len(kb.clauses)} clauses, {len(syms)} symbols, 2^{len(syms)} models')
    print('  ' + try_inference('pl_resolution', lambda: pl_resolution(kb, board_query),
                               mem_limit=MEMORY_BUDGET))
    print('  ' + try_inference('tt_entails',
                               lambda: tt_entails(associate('&', kb.clauses), board_query)))
    print()

# Soundness check on the 2x2: a wrong value has to come back False, not just
# "no answer". If this printed True the encoding would be the problem, not size.
tiny_kb = build_general_kb(2, 1, 2, tiny_givens)
print(try_inference('pl_resolution on Is2_2_2 (wrong value)',
                    lambda: pl_resolution(tiny_kb, atom('Is', 2, 2, 2)),
                    mem_limit=MEMORY_BUDGET))

2x2 board (1x2 boxes): 21 clauses, 8 symbols, 2^8 models
  pl_resolution: returned True in 0.02s
  tt_entails: returned True in 0.00s

4x4 board (2x2 boxes): 412 clauses, 64 symbols, 2^64 models
  pl_resolution: gave up, still running after 20s
  tt_entails: gave up, still running after 20s

pl_resolution on Is2_2_2 (wrong value): returned False in 0.03s


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


Ran both algorithms on the KB from `build_general_kb`. Neither one solves a single cell of the 9x9 puzzle.

**On the real puzzle.** The general KB comes out to **13,233 clauses over 1,458 symbols**. The query is `Is1_1_1`: cell (1,1) is empty in this puzzle and its solved value is 1, so the correct answer is `True`, and it has to be inferred rather than looked up. That is a single query, though: solving the puzzle the way task (a) describes it means repeating this for every empty cell, up to 81 x 9 = 729 entailment queries. Neither algorithm finishes even the first one.

- `pl_resolution` never returned. It does not run out of time first, it runs out of space. `pl_resolution` builds the complete list of clause pairs before resolving any of them (`pairs = [(clauses[i], clauses[j]) ...]` in `logic_.py`), and with 13,233 clauses that is **87,549,528 pairs in the first round alone**. It hit our 800 MB budget about 4 seconds in, still inside round one. The cost argument is simple: one round compares every pair of clauses, so `O(N^2)` for `N` clauses -- and every resolvent found is added back into the clause set, so `N` is larger on the next round and that `N^2` is measured against the larger `N`. Run without a memory guard we measured it go past **4 GB in roughly 10 seconds**, still inside round one -- which is why the cell above watches memory and stops the thread, rather than just timing out and leaving it running.
- `tt_entails` never returned either. It needs 1,458 nested calls just to reach a *single* complete model, and there are 2^1458 of them -- about 10^439. It was still going when we stopped it at 20s. The cost argument: it checks exactly `2^s` models for `s` symbols, so each extra symbol doubles the work -- and every one of the 81 cells contributes 9 symbols.

  (Small detail: run outside Jupyter this fails even earlier. Python's default recursion limit is 1,000, which is less than 1,458, so `tt_check_all` raises `RecursionError` before it finishes even one model. The Jupyter kernel raises the limit to 3,000, so here it clears the stack and just never finishes instead.)

**Where the wall actually is.** Same builder, same two algorithms, smaller boards:

| board | clauses | symbols | `pl_resolution` | `tt_entails` |
|---|---|---|---|---|
| 2x2 (1x2 boxes) | 37 | 16 | `True` in 0.13s | `True` in 0.34s |
| 4x4 (2x2 boxes) | 540 | 128 | no answer in 20s | no answer in 20s |
| 9x9 (3x3 boxes) | 13,233 | 1,458 | no answer, 936 MB | no answer in 20s |

On the 2x2 both return the right answer, and `pl_resolution` correctly returns `False` for a wrong value (`Is2_2_2`, in 0.61s). So the encoding is fine and both algorithms are doing exactly what they promise -- size is the only thing that breaks them. But the wall is already there at **4x4**, a board with just 540 clauses, and the real puzzle is about 24 times bigger than that.

One last detail: `build_general_kb` also tells `Is_rcv <=> ~Not_rcv` for every cell and value, which doubles the symbol count from 729 to 1,458. Commenting that block out halves the exponent, but 2^729 is still around 10^219, so it changes nothing that matters.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [45]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [46]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")


# Add in code to time solve_full_grid_fc_one_pass
t0 = time.time()
solved = solve_full_grid_fc_one_pass(n, box_h, box_w, givens)
fc_op_time = time.time() - t0
assert solved == puzzle['solution']

print(f"solve_full_grid_fc_one_pass: {fc_op_time:.2f}s")

solve_full_grid_fc: 16.73s
solve_full_grid_bc: 34.26s
solve_full_grid_fc_one_pass: 11.47s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

### 1(a) General KB strategy
Since there are no structural restrictions, I limited the symbol types used to only $Is$, using both the positive ($Is$) and negative ($\neg Is$) variants. This eliminated the need for maintaining 2 symbol states to represent each cell, and thereby reduced the total number of clauses in the KB. The standard sudoku rules are encoded based on the following principles:

#### 1. At least 1 value per cell

This rule states that, out of 9 possible values ($Is_{r,c,1}$ to $Is_{r,c,9}$) at-least one value must be true. This is directly converted into a disjunction of all literals being true for each cell. This will ensure that at least one of the $Is$ symbols for every cell turns out to be True to hold the whole statement as True.

$$
Is_{1,1,1} \lor Is_{1,1,2} \lor Is_{1,1,3} \lor \cdots \lor Is_{1,1,8} \lor Is_{1,1,9}
$$

$$
Is_{1,2,1} \lor Is_{1,2,2} \lor Is_{1,2,3} \lor \cdots \lor Is_{1,2,8} \lor Is_{1,2,9}
$$

$$
\vdots
$$

This is repeated for every cell.

#### 2. At most one value per cell

This rule states that, out of 9 possible values ($Is_{r,c,1}$ to $Is_{r,c,9}$) only one value must be true. This can be rendered as an implication. If $Is_{r,c,1}$ is true then it implies none of the other values (i.e, $Is_{r,c,2}$ to $Is_{r,c,9}$) can be true. Similarly is $Is_{r,c,2}$ is true then it implies $Is_{r,c,1}$, $Is_{r,c,3}$ to $Is_{r,c,9}$ cannot be true. The left hand side of the implication will be a single symbol representing one value for the given cell. The right side of the implication will be the conjunction of negation of remaining values as shown below.

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,2} \land \neg Is_{1,1,3} \land \neg Is_{1,1,4} \land \neg Is_{1,1,5} \land \neg Is_{1,1,6} \land \neg Is_{1,1,7} \land \neg Is_{1,1,8} \land \neg Is_{1,1,9}
$$

Since the right hand side is a conjunction of multiple variables, it can be simplified into individual implications between each pair of values like below.

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,2}
$$

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,3}
$$

$$
\vdots
$$

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,9}
$$

This is repeated for every cell and every possible value it can take.

#### 3. Row uniqueness, Column uniqueness, Box uniqueness

These conditions follow the same pattern as the one used for "At most one value per cell".

Row uniqueness - If a value is locked for a row, it cannot appear in any other cell in that row.

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,2,1}, \quad Is_{1,1,1} \Rightarrow \neg Is_{1,3,1}, \quad \ldots \quad Is_{1,1,1} \Rightarrow \neg Is_{1,9,1}
$$

$$
Is_{1,1,2} \Rightarrow \neg Is_{1,2,2}, \quad Is_{1,1,2} \Rightarrow \neg Is_{1,3,2}, \quad \ldots \quad Is_{1,1,2} \Rightarrow \neg Is_{1,9,2}
$$

$$
\vdots
$$

$$
Is_{1,1,9} \Rightarrow \neg Is_{1,2,9}, \quad Is_{1,1,9} \Rightarrow \neg Is_{1,3,9}, \quad \ldots \quad Is_{1,1,9} \Rightarrow \neg Is_{1,9,9}
$$

This repeats for every row and value combination.

Column uniqueness - If a value is locked for a column, it cannot appear in any other cell in that column.

$$
Is_{1,1,1} \Rightarrow \neg Is_{2,1,1}, \quad Is_{1,1,1} \Rightarrow \neg Is_{3,1,1}, \quad \ldots \quad Is_{1,1,1} \Rightarrow \neg Is_{9,1,1}
$$

$$
Is_{1,1,2} \Rightarrow \neg Is_{2,1,2}, \quad Is_{1,1,2} \Rightarrow \neg Is_{3,1,2}, \quad \ldots \quad Is_{1,1,2} \Rightarrow \neg Is_{9,1,2}
$$

$$
\vdots
$$

$$
Is_{1,1,9} \Rightarrow \neg Is_{2,1,9}, \quad Is_{1,1,9} \Rightarrow \neg Is_{3,1,9}, \quad \ldots \quad Is_{1,1,9} \Rightarrow \neg Is_{9,1,9}
$$

This repeats for every column and value combination.

Box uniqueness - If a value is locked for a box, it cannot appear in any other cell in that box.

$$
Is_{1,1,1} \Rightarrow \neg Is_{2,2,1} \quad \ldots \quad \text{with every cell within its box}
$$

This repeats for every box and value combination.





### 1(b) Definite Knowledge Base Strategy

The definite knowledge base uses two families of propositional symbols:

- $Is_{r,c,v}$: cell $(r,c)$ contains value $v$.
- $Not_{r,c,v}$: value $v$ has been eliminated from cell $(r,c)$.

Although its name begins with “Not”, $Not_{r,c,v}$ is a **positive propositional atom**, rather than the logical negation $\neg Is_{r,c,v}$. This allows elimination rules to be expressed as definite clauses: implications with a conjunction of positive atoms as premises and one positive atom as the conclusion.

The encoding uses the following facts and rules.

#### 1. Every cell has at least one value

The general constraint

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}
$$

contains multiple positive literals, so it cannot be inserted directly as a definite clause.

Instead, the encoding uses **last-candidate rules**. For each cell and each possible value $v$, if every other value has been eliminated, the solver infers that the cell contains $v$:

$$
\left(
\bigwedge_{\substack{u=1 \\ u \ne v}}^{n} Not_{r,c,u}
\right)
\Rightarrow Is_{r,c,v}.
$$

For example, in a 9×9 puzzle:

$$
Not_{1,1,2} \land Not_{1,1,3} \land \cdots \land Not_{1,1,9}
\Rightarrow Is_{1,1,1}.
$$

One such rule is generated for every candidate value in every cell.

These rules capture a consequence of the “at least one” constraint, rather than expressing the complete constraint equivalently: they infer a value only when all alternatives have been eliminated.

#### 2. Every cell has at most one value

When a cell is known to contain $v$, every different value $u$ is eliminated from that cell:

$$
Is_{r,c,v} \Rightarrow Not_{r,c,u}
\qquad \text{for every } u \ne v.
$$

For example:

$$
Is_{1,1,1} \Rightarrow Not_{1,1,2}.
$$

A separate implication is generated for each alternative value, keeping every conclusion a single positive atom.

#### 3. No two cells in the same row hold the same value

If cell $(r,c)$ contains $v$, that value is eliminated from every other cell in row $r$:

$$
Is_{r,c,v} \Rightarrow Not_{r,c',v}
\qquad \text{for every } c' \ne c.
$$

#### 4. No two cells in the same column hold the same value

Similarly, a known value is eliminated from every other cell in the same column:

$$
Is_{r,c,v} \Rightarrow Not_{r',c,v}
\qquad \text{for every } r' \ne r.
$$

#### 5. No two cells in the same box hold the same value

A known value is also eliminated from every other cell in its box:

$$
Is_{r,c,v} \Rightarrow Not_{r',c',v},
$$

where $(r',c')$ is a different cell in the same box. The implementation avoids adding duplicate elimination rules for box cells already covered by the row or column rules.

#### 6. Given cells hold their stated values

Each given is added directly as a positive fact. For example, if cell $(1,1)$ initially contains 5, the KB includes:

$$
Is_{1,1,5}.
$$

These facts provide the starting point for inference.

#### Additional last-location rules

The implementation also includes rules that infer a value when all other locations for that value have been eliminated within a row, column, or box. For example, if every other cell in a row rules out 5, the remaining cell must contain 5. These are commonly called **hidden-single deductions**.

Together, the rules create a deduction cycle: known values produce eliminations, eliminations establish new values, and those values produce further eliminations. This supports forward and backward chaining, although puzzles requiring reasoning beyond the encoded rules may remain partially unsolved.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

No -- neither one is usable as the default, and the experiment above is the short version of why: on the 9x9 puzzle neither algorithm answers even one cell.

The guarantees are real, but they are guarantees about *correctness*, not about *cost*. Soundness means the algorithm never infers anything that is not entailed; completeness means it can eventually derive everything that is. Neither says "eventually" is a useful amount of time.

**Model checking (`tt_entails`).** It decides entailment by enumerating every assignment of truth values to the symbols and checking that the query holds in every row where the KB holds. That is `2^s` models for `s` symbols -- exponential in the number of variables, as in the lecture. Our KB has 1,458 symbols, so 2^1458, roughly 10^439 models. For scale, the observable universe has about 10^80 atoms. This is not slow, it is impossible, and a faster machine does not help: every extra empty cell multiplies the work by another 2^9. The implementation also recurses once per symbol, so it needs 1,458 nested calls just to reach one complete model -- past Python's default recursion limit before it has checked a single one.

**Resolution-refutation (`pl_resolution`).** It adds `~query` to the clause set and repeatedly resolves every pair of clauses, looking for the empty clause. One round costs `O(N^2)` for `N` clauses. The trouble is that `N` does not stay put: every resolvent gets added back into the clause set, so each round starts with more clauses than the last and the next `N^2` is measured against the bigger `N`. With 13,233 starting clauses, round one is already 87.5 million pairs -- and because the implementation materialises all of them before resolving any, we ran out of memory inside round one rather than finishing it. Clause *length* pushes the same way: resolving two long clauses gives another long clause, and longer clauses resolve with more partners.

**What that means for Sudoku.** The difficulty here is not deep reasoning, it is scale -- an `n x n` board needs `n^3` symbols and `O(n^4)` constraint clauses. Both algorithms are fully general: they make no use of the fact that almost every Sudoku constraint is a simple elimination. So the sensible move is the one the rest of this assignment takes -- restrict the KB to Horn clauses and use forward or backward chaining, which run in time linear in the size of the KB instead of exponential in the number of symbols. We give up arbitrary disjunctions (see Question 1(b)) and get back a solver that finishes.

That said, it is worth being precise about what is being given up. On the 2x2 board both algorithms answered correctly, including returning `False` for a wrong value. Resolution is still the right tool when you genuinely need general clauses, or when you need to prove something *unsatisfiable* -- which chaining over a definite KB cannot do at all, since such a KB is always satisfiable. It is simply the wrong default at this size.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**
I started designing the algorithm based on the pseudocode provided in the lecture slides. Soon it was clear that was not enough. I ran into recursive loop problems and time complexity issues. To fix that, 3 main things were added on top of the provided lecture implementation.
1. `visited`: This set helps to check if a current cell has already been seen in the current recursive stack of checks (so that new  recursive stacks can be reevaluated given new information), and if it has the results are not checked again (to prevent recursive loops if 2 variables are codependent).
2. `failed`: This is purely a time saving set, where every query which is bound to fail (not go anywhere meaningful) is added to this set. It makes the search much much more efficient. The set is cleared innocently everytime a new fact is added to the kb, since past routes that don't go anywhere might have an answer now
3. `kb_cleanup`: This function removes redundant knowledge from the kb, providing efficiency to the solver. Facts that are irrelevant since we know either `Is_r_c_v` or `Not_r_c_v`, any code utilizing its opposite evaluates to False hence is not useful. This helps massively after the first pass or first few passes where much more facts are known.
---
### Modified BC Pseudocode
```text
function pl_bc_entails?(KB,q) returns true or false
    inputs: KB, the knowledge base, a set of propositional definite clauses
            q: the query, a propositional symbol

if q matches a fact, then return true
if q has already been seen in the current recursion stack, then return False
if ~q is known in the kb, then return False and add q to failed set
if there is no clause with a consequent that matches q, then return False, remove from visited set and add to Failed set

for each clause c in KB where q is in c.CONCLUSION do
    count = Number of symbols in c.PREMISE
    for all symbols in c.PREMISE do
        if pl_bc_entails?(KB,q)
            count -= 1
        else break
    if count == 0
        if q not in kb
            add q to kb
            kb_cleanup(kb,q)
            clear the failed set
        remove q from visited
        return true
remove q from visited
add q to failed
return false
```


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

A Naked Pair occurs when two cells in the same row, column, or box have the same two remaining candidate values.

For example, suppose cells $(1,1)$ and $(1,4)$ can each contain only 2 or 7. Because they share a row, they cannot both contain the same value. Therefore, one must contain 2 and the other must contain 7, although we may not yet know which is which.

Consequently, 2 and 7 can be eliminated from every other cell in row 1.

#### ~~Representing the premises using `Not` atoms~~

~~The existing vocabulary is sufficient:~~

==Using the existing syntax:==

- $Is_{r,c,v}$ means cell $(r,c)$ contains value $v$.
- $Not_{r,c,v}$ means value $v$ has been eliminated from cell $(r,c)$.

To express that a cell is restricted to candidates $a$ and $b$, we require explicit $Not$ facts for every value outside $\{a,b\}$.

For two cells $(r,c_1)$ and $(r,c_2)$ in the same row, the combined premises are:

$$
\bigwedge_{u \in \{1,\ldots,n\}\setminus\{a,b\}}
\left(
Not_{r,c_1,u} \land Not_{r,c_2,u}
\right).
$$

These premises establish that both cells are restricted to the same pair of values.

Importantly, this uses established elimination facts. It does not treat the absence of an $Is$ fact as evidence that a value is impossible.

#### Definite-clause encoding

For every other column $c_3 \notin \{c_1,c_2\}$ in row $r$, generate two separate implications:

$$
\left[
\bigwedge_{u \in \{1,\ldots,n\}\setminus\{a,b\}}
\left(
Not_{r,c_1,u} \land Not_{r,c_2,u}
\right)
\right]
\Rightarrow Not_{r,c_3,a}.
$$

$$
\left[
\bigwedge_{u \in \{1,\ldots,n\}\setminus\{a,b\}}
\left(
Not_{r,c_1,u} \land Not_{r,c_2,u}
\right)
\right]
\Rightarrow Not_{r,c_3,b}.
$$

Each implication is a valid definite clause because its premises are a conjunction of positive atoms and its conclusion is one positive atom. Although the symbols begin with `Not`, they are positive propositional atoms rather than logical negations.

The same construction applies to any two cells sharing a column or box, with conclusions eliminating the pair's values from other cells in that shared unit.

#### Example

Suppose cells $(1,1)$ and $(1,4)$ are restricted to $\{2,7\}$ in a 9×9 puzzle.

The KB must establish that both cells exclude every value in:

$$
E = \{1,3,4,5,6,8,9\}.
$$

Here, $E$ is only shorthand for a set of digits, not an additional propositional symbol.

For another cell in the row, such as $(1,6)$, add:

$$
\left[
\bigwedge_{u \in E}
\left(
Not_{1,1,u} \land Not_{1,4,u}
\right)
\right]
\Rightarrow Not_{1,6,2}.
$$

$$
\left[
\bigwedge_{u \in E}
\left(
Not_{1,1,u} \land Not_{1,4,u}
\right)
\right]
\Rightarrow Not_{1,6,7}.
$$

Each rule has 14 premise atoms: seven eliminated values for each of the two cells.

Equivalent rules are generated for every other cell in row 1. Once the premises are established, forward chaining can derive the additional eliminations. These may then trigger existing last-candidate rules and reveal further cell values.

#### Consequences of adding these rules

**Stronger deductions.**  
The solver can make eliminations even when neither cell in the pair has a known value. This can allow progress when the original elimination and single-candidate rules have stalled.

~~**No new vocabulary or inference algorithm is required.**  
The rules use the existing `Not` atoms and remain definite clauses. They can therefore be processed by the same forward- or backward-chaining algorithms.~~

**A larger knowledge base.**  
A general encoding must consider possible pairs of cells, pairs of values, and target cells in every row, column, and box. Generating all these rules in advance can substantially increase memory usage.

**A performance trade-off.**  
The additional rules may help solve more cells, but they also increase rule-processing work. Indexing rules by their premises can reduce unnecessary scanning during forward chaining.

~~**Soundness depends on the intended Sudoku interpretation.**  
The deduction follows from the requirements that each cell has a value and values cannot repeat within a unit. Provided the elimination facts are sound and the puzzle is consistent, the resulting eliminations are sound.~~

**The solver is still not complete for every Sudoku puzzle.**  
Adding Naked Pairs extends the encoded reasoning strategies, but some puzzles require other techniques. Forward chaining remains complete for the definite KB: it derives everything implied by those rules, which ~~need not include~~ does not necessarily result in a full Sudoku solution.

Therefore, Naked Pairs demonstrates that definite clauses can express more than simple single-cell deductions. The practical limitation is the reasoning encoded in the rules and the cost of storing and applying them, rather than an inability to represent this technique in Horn logic.

### ~~Limitations of using Definite Clause Knowledge Base~~
~~As mentioned above, even though deductions using forward/backward chaining on this knowledge base are complete, we are still unable to solve some sudoku puzzles. Comparing the general representation of the constraint "Every cell has at least one value from {1, . . . , n}.",~~
$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}
$$
to the definite clause representation,
$$
\left(
\bigwedge_{\substack{u=1 \\ u \ne v}}^{n} Not_{r,c,u}
\right)
\Rightarrow Is_{r,c,v}
$$

~~we can see that the definite clause representation does not strictly enforce that every cell has a value from 1 to n unless we have eliminated n-1 possibilities. In the case where there could be 2 or more possible values after complete forward chaining inference, additional mechanisms such as backtracking are needed to "guess" values and check the results.~~


### Limitations of Using a Definite-Clause Knowledge Base

Although forward and backward chaining are complete for atomic queries over a definite-clause knowledge base, this does not guarantee that they can solve every Sudoku puzzle. Their completeness applies to the encoded rules, which do not preserve all the logical consequences of the full Sudoku constraints.

For example, the general representation of “Every cell has at least one value from $\{1,\ldots,n\}$” is

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}.
$$

In the definite-clause encoding, this is replaced by a family of last-candidate rules, one for each possible value $v$:

$$
\left(
\bigwedge_{\substack{1 \le u \le n \\ u \ne v}}
Not_{r,c,u}
\right)
\Rightarrow Is_{r,c,v}.
$$

These rules allow a value to be inferred once all $n-1$ alternatives have been eliminated. However, they are not logically equivalent to the original disjunction: they do not enforce that some value must hold when multiple candidates remain. Here, each $Not_{r,c,u}$ is a separate atom representing an established elimination, rather than an automatic logical negation of $Is_{r,c,u}$.

Consequently, forward chaining may reach a fixed point—where no additional facts can be derived—while some cells remain unresolved. Complete backward chaining over the same KB cannot prove those missing assignments either. Further progress requires reasoning beyond the existing rules, such as additional deduction rules or backtracking. Backtracking tentatively assigns candidate values, explores their consequences, and reverses choices that lead to contradictions. Thus, the limitation lies in the encoded reasoning rules rather than in the completeness of chaining.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

#### 5(a) When is backward chaining faster than forward chaining
Backward chaining can perform faster when the query relevant subset is much smaller than the whole KB or when most of the symbols within the query relevant subset are pre-established facts. In such a scenario, Backward chaining scans all clauses in the KB only on those query subset symbols and when a symbol is a pre-established fact, there is no recursion required to determine its state. On the other hand, forward chaining will start with all the known facts and keep scanning through all the clauses and resolve facts until it encounters the query symbol. The demonstration in cell #47 shows a scenario where backward chaining query performs faster than forward chaining.

#### 5(b) When backward chaining performs worse than forward chaining
On a full grid evaluation, there are repeated cell value queries. This results in repeated recursive scans even though the query relevant subset may be small. This is partially addressed in our solution by cleaning up failed cell values and adding deduced facts back into the KB. However, it does not totally eliminate the recursive searches. Forward chaining along with feeding deduced facts back into the KB is better for a full grid evaluation as it helps in deducing multiple facts over each of its scans. We wrote another enhanced version of forward chaining for solving the sudoku puzzle - solve_full_grid_fc_one_pass. This is a one pass forward chaining algorithm which eliminates all the repeated work in solve_full_grid_fc by continuining forward chaining until the values for all the cells are deduced. Cell #48 shows the performance of all three algorithms on full grid puzzles.

In [ ]:
# Experiment to prove backward chaining can be faster than forward chaining
from IPython.display import Markdown, display

sample_puzzle = puzzle_pool[0]
sample_solution = sample_puzzle['solution']
query_cell = (5, 5)
query_value = sample_solution[query_cell]

# Populate every other cell in the target row and target column.
sample_givens = {}
for c in range(1, n + 1):
    if c != query_cell[1]:
        sample_givens[(query_cell[0], c)] = sample_solution[(query_cell[0], c)]

for r in range(1, n + 1):
    if r != query_cell[0]:
        sample_givens[(r, query_cell[1])] = sample_solution[(r, query_cell[1])]

assert query_cell not in sample_givens
assert len(sample_givens) == (2 * n) - 2
query = atom('Is', query_cell[0], query_cell[1], query_value)

print(f'Synthetic puzzle: {len(sample_givens)} givens; target {query_cell} is empty')
for r in range(1, n + 1):
    print([str(sample_givens.get((r, c), '.')) for c in range(1, n + 1)])

# Use separate but equivalent KBs. KB construction is intentionally not timed.
single_query_fc_kb = build_definite_kb(n, box_h, box_w, sample_givens)
single_query_bc_kb = build_definite_kb(n, box_h, box_w, sample_givens)

start_time = time.perf_counter()
fc_result = pl_fc_entails(single_query_fc_kb, query)
fc_query_time = time.perf_counter() - start_time
assert fc_result is True

start_time = time.perf_counter()
bc_result = pl_bc_entails(single_query_bc_kb, query)
bc_query_time = time.perf_counter() - start_time
assert bc_result is True

single_query_table = [
    '| Measurement | Result |',
    '|---|---:|',
    f'| Number of givens | {len(sample_givens)} |',
    f'| Query | `{query}` |',
    f'| Total KB clauses | {len(single_query_fc_kb.clauses):,} |',
    f'| FC entailment time | {fc_query_time:.4f} s |',
    f'| BC entailment time | {bc_query_time:.4f} s |',
    f'| BC speedup over FC | {fc_query_time / bc_query_time:.2f}x |',
]

display(Markdown('\n'.join(single_query_table)))

Synthetic puzzle: 16 givens; target (5, 5) is empty
['.', '.', '.', '.', '8', '.', '.', '.', '.']
['.', '.', '.', '.', '1', '.', '.', '.', '.']
['.', '.', '.', '.', '6', '.', '.', '.', '.']
['.', '.', '.', '.', '9', '.', '.', '.', '.']
['3', '6', '9', '8', '.', '2', '5', '1', '4']
['.', '.', '.', '.', '3', '.', '.', '.', '.']
['.', '.', '.', '.', '4', '.', '.', '.', '.']
['.', '.', '.', '.', '5', '.', '.', '.', '.']
['.', '.', '.', '.', '2', '.', '.', '.', '.']


| Measurement | Result |
|---|---:|
| Number of givens | 16 |
| Query | `Is5_5_7` |
| Total KB clauses | 18,652 |
| FC entailment time | 0.8497 s |
| BC entailment time | 0.2662 s |
| BC speedup over FC | 3.19x |

In [48]:
# Experiment to check performance of all 3 algorithms on all given puzzles
from IPython.display import Markdown, display

algorithms = [
    ('solve_full_grid_fc_one_pass', solve_full_grid_fc_one_pass),
    ('solve_full_grid_bc', solve_full_grid_bc),
    ('solve_full_grid_fc', solve_full_grid_fc),
]

assert len(puzzle_pool) == 5, f'Expected 5 puzzles, found {len(puzzle_pool)}'
benchmark_rows = []

for puzzle_number, current_puzzle in enumerate(puzzle_pool, start=1):
    timings = {}
    answers = {}

    for algorithm_name, solver in algorithms:
        start_time = time.perf_counter()
        solved = solver(n, box_h, box_w, dict(current_puzzle['givens']))
        elapsed_time = time.perf_counter() - start_time

        assert solved == current_puzzle['solution'], (
            f'Puzzle {puzzle_number}: {algorithm_name} returned an incorrect solution'
        )

        timings[algorithm_name] = elapsed_time
        answers[algorithm_name] = solved

    assert (
        answers['solve_full_grid_fc_one_pass']
        == answers['solve_full_grid_bc']
        == answers['solve_full_grid_fc']
    ), f'Puzzle {puzzle_number}: the three algorithms returned different answers'

    benchmark_rows.append({
        'Puzzle': puzzle_number,
        'Given cells': current_puzzle['given_count'],
        'solve_full_grid_fc_one_pass (s)': timings['solve_full_grid_fc_one_pass'],
        'solve_full_grid_bc (s)': timings['solve_full_grid_bc'],
        'solve_full_grid_fc (s)': timings['solve_full_grid_fc'],
    })

    print(f'Puzzle {puzzle_number}: all three solutions verified')

table_lines = [
    '| Puzzle | Given cells | FC one pass (s) | BC (s) | FC (s) |',
    '|---:|---:|---:|---:|---:|',
]

for row in benchmark_rows:
    table_lines.append(
        f"| {row['Puzzle']} | {row['Given cells']} "
        f"| {row['solve_full_grid_fc_one_pass (s)']:.4f} "
        f"| {row['solve_full_grid_bc (s)']:.4f} "
        f"| {row['solve_full_grid_fc (s)']:.4f} |"
    )

display(Markdown('\n'.join(table_lines)))

Puzzle 1: all three solutions verified
Puzzle 2: all three solutions verified
Puzzle 3: all three solutions verified
Puzzle 4: all three solutions verified
Puzzle 5: all three solutions verified


| Puzzle | Given cells | FC one pass (s) | BC (s) | FC (s) |
|---:|---:|---:|---:|---:|
| 1 | 30 | 11.4202 | 36.2041 | 18.6572 |
| 2 | 33 | 11.4053 | 22.0560 | 12.2353 |
| 3 | 36 | 11.2016 | 18.8318 | 15.2084 |
| 4 | 39 | 11.5209 | 15.4399 | 11.5761 |
| 5 | 42 | 10.9949 | 12.4233 | 11.2893 |

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
